# Aralin 13 - Memorya ng Ahente gamit ang Cognee Knowledge Graphs


## Setup

Ipinapakita ng notebook na ito kung paano gumawa ng isang matalinong **coding assistant** na may persistent memory gamit ang [**Cognee**](https://www.cognee.ai/) knowledge graphs at ang **Microsoft Agent Framework** (MAF).

Binabago ng Cognee ang hindi estrukturadong teksto sa isang estrukturadong, queryable knowledge graph na sinusuportahan ng vector embeddings — nagbibigay sa iyong agent ng mayamang, relasyon-na-alam na pangmatagalang memorya.

### Ano ang Matututuhan Mo
1. **Gumawa ng Knowledge Graphs**: I-transform ang mga developer profile at pinakamahuhusay na praktis sa estrukturadong, queryable na kaalaman.
2. **Pagsamahin ang Cognee sa MAF**: Gamitin ang `@tool` na mga function para payagan ang isang MAF agent na i-query ang knowledge graph ng Cognee.
3. **Session-Aware na Mga Usapan**: Panatilihin ang konteksto sa maraming tanong sa iisang session.
4. **Pangmatagalang Memorya**: I-persist ang importanteng kaalaman sa mga session at kunin ito sa mga bagong pag-uusap.

### Mga Kinakailangan
- Python 3.9+
- Redis na tumatakbo sa lokal (`docker run -d -p 6379:6379 redis`) para sa pamamahala ng session
- Isang LLM API key (hal. OpenAI) — itakda ang `LLM_API_KEY` sa `.env`
- `CACHING=true` sa `.env` (kinakailangan para sa mga session ng Cognee)
- Isang Azure AI Foundry project na may inilunsad na chat model
- `AZURE_AI_PROJECT_ENDPOINT` at `AZURE_AI_MODEL_DEPLOYMENT_NAME` sa `.env`
- Azure CLI na naka-authenticate (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Mga Uri ng Memorya ng Ahente

Tinutuklas ng notebook na ito ang parehong tatlong uri ng memorya mula sa pangunahing Lesson 13 notebook, ngunit ginagamit ang Cognee bilang long-term memory backend:

| Uri ng Memorya | Mekanismo | Haba ng Buhay |
|---|---|---|
| **Pang-gawain** | `agent.create_session()` (MAF) | Isang pag-uusap |
| **Pang-maikling termino** | Cognee session cache (Redis) | Isang session |
| **Pang-matagalang termino** | Cognee knowledge graph + vectors | Permanenteng |

### Arkitektura ng Memorya ng Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Ihanda ang Cognee Storage


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Bahagi 1 — Pagtatayo ng Knowledge Base

Tumatanggap kami ng tatlong uri ng datos upang makalikha ng komprehensibong knowledge base para sa aming coding assistant:

1. **Developer Profile** — personal na kadalubhasaan at teknikal na background
2. **Python Best Practices** — ang Zen ng Python na may praktikal na mga gabay
3. **Historical Conversations** — mga nakaraang Q&A sessions sa pagitan ng mga developer at AI assistants


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## I-visualize ang Knowledge Graph

Maaaring mag-render ang Cognee ng isang interactive na HTML na biswal na representasyon ng mga entidad at ugnayan na kanyang nakuha.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Enrich Memory with Memify

`memify()` sumusuri sa knowledge graph at bumubuo ng matatalinong patakaran — tinutukoy ang mga pattern, pinakamahuhusay na gawain, at mga relasyon sa pagitan ng mga konsepto.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Part 2 — MAF Agent gamit ang Cognee Tools

Ngayon ay gagawa tayo ng isang MAF agent na maaaring mag-query sa knowledge graph ng Cognee gamit ang mga `@tool` na function. Pinapayagan nito ang agent na gamitin ang buong kapangyarihan ng graph-aware semantic search habang pinananatili ang konteksto ng pag-uusap sa pamamagitan ng mga sesyon.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Working Memory with Sessions

Ang `AgentSession` (na nilikha sa pamamagitan ng `agent.create_session()`) ay nagbibigay ng working memory sa loob ng isang session. Maaaring balikan ng ahente ang mga naunang mensahe habang nag-uusisa rin sa pangmatagalang knowledge graph ng Cognee.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Bagong Session — Nanatili ang Pangmatagalang Alaala

Ang pagsisimula ng bagong session ay naglilinis ng working memory, ngunit ang Cognee knowledge graph ay nananatiling magagamit. Maaari nitong makuha ang parehong pangmatagalang kaalaman sa isang ganap na bagong pag-uusap.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Buod

Sa notebook na ito ay bumuo ka ng isang coding assistant na pinagsasama ang **MAF's working memory** (`agent.create_session()`) sa **Cognee's long-term knowledge graph**.

### Mga Natutunan Mo
1. **Pagbuo ng knowledge graph**: Kinukuha ng Cognee ang hindi istrukturadong teksto at bumubuo ng graph + vector memory.
2. **Pagyaman ng graph gamit ang memify**: Mga nakuha na katotohanan at mas mayamang relasyon sa ibabaw ng umiiral mong graph.
3. **Integrasyon ng MAF + Cognee**: Ang mga `@tool` na function ay nagpapahintulot sa isang MAF agent na natural na magtanong sa graph ng Cognee.
4. **Working memory + long-term memory**: Ang `AgentSession` (sa pamamagitan ng `agent.create_session()`) ay nagbibigay ng session context habang ang Cognee ay nagbibigay ng persistent na kaalaman.
5. **Pinag-filter na paghahanap gamit ang NodeSets**: Targetin ang mga partikular na subsets ng knowledge graph (hal. mga prinsipyo lamang).

### Pangunahing Mga Dapat Tandaan
- Ang **Cognee** ay ginagawang estrukturado at may kamalayan sa relasyon ang raw na teksto — mas makapangyarihan kaysa sa simpleng vector store.
- Ang mga **`@tool` function** ay nag-uugnay ng malinis sa mga MAF agent at mga external na knowledge system.
- Ang **`AgentSession`** (sa pamamagitan ng `agent.create_session()`) ay naghihiwalay ng konteksto ng bawat usapan mula sa matagal na kaalaman.
- Ang parehong knowledge graph ay nagsisilbi sa maraming sesyon at mga agent.

### Mga Aplikasyon sa Totoong Mundo
- **Developer copilots**: Pag-review ng code, pagsusuri ng insidente, mga architecture assistant
- **Customer-facing copilots**: Mga support agent sa ibabaw ng mga product docs, FAQs, at CRM notes
- **Internal expert copilots**: Mga assistant sa polisiya, legal, o seguridad na nagrereason sa mga gabay
- **Unified data layers**: Pagsamahin ang estrukturado at hindi estrukturadong data sa isang queryable graph

### Mga Susunod na Hakbang
- Subukan ang temporal awareness sa Cognee
- Magdefine ng OWL ontology para sa domain-specific na kalidad ng graph
- Magdagdag ng user feedback loops upang mapabuti ang retrieval sa pagdaan ng panahon
- I-scale sa multi-agent systems na nagbabahagi ng iisang Cognee memory layer


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Paunawa**:  
Ang dokumentong ito ay isinalin gamit ang AI translation service na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't sinisikap naming maging tumpak, pakatandaan na ang mga awtomatikong salin ay maaaring maglaman ng mga pagkakamali o kamalian. Ang orihinal na dokumento sa kanyang orihinal na wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na salin ng tao. Hindi kami mananagot sa anumang hindi pagkakaunawaan o maling interpretasyon na maaaring magmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
